In [ ]:
import torch
import pandas as pd
import time
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "distil-whisper/distil-large-v2"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
print()

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    torch_dtype=torch_dtype,
    device=device,
    model_kwargs={"attn_implementation": "flash_attention_2"},
)

In [ ]:
file_name =  "/home/aimore/GIT/transcription/data/audio_samples/long_sample.mp3"
batch_size = 256
generate_kwargs = {"task": "transcribe", "language": "en"}
ts = True

outputs = pipe(
            file_name,
            chunk_length_s=30,
            batch_size=batch_size,
            generate_kwargs=generate_kwargs,
            return_timestamps=ts,
            # stride_length_s=(5, 3)
        )

In [ ]:
pd.set_option('display.max_columns',200, 'display.max_colwidth', 200, 'display.max_rows',15, 'display.min_rows',15)

df = pd.DataFrame(outputs['chunks'])

df['length'] = df.timestamp.apply(lambda x: x[1] - x[0])

def convert(sec):
    current_time = time.strftime("%H:%M:%S", time.gmtime(sec))
    return current_time

df['time'] = df.timestamp.apply(lambda sec: convert(sec[0]))
df
# df.sort_values('length')
# df.text.value_counts()